# Random Forest Baseline & Socio-Demographic Importance

This notebook fits a Random Forest regression model on spatiotemporal daily COVID-19 case features to isolate socio-demographic and kinetic feature importances.


In [7]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor

print('✅ Core libraries imported successfully!')


✅ Core libraries imported successfully!


In [8]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Pure NumPy Metric Calculations (No seaborn / No sklearn.metrics)
def calc_rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

def calc_mae(y_true, y_pred):
    return float(np.mean(np.abs(y_true - y_pred)))

def calc_pr_auc(y_true_binary, y_pred_scores):
    thresholds = np.sort(np.unique(y_pred_scores))[::-1]
    precisions, recalls = [1.0], [0.0]
    total_positives = np.sum(y_true_binary == 1)
    if total_positives == 0:
        return 0.0
    for th in thresholds:
        tp = np.sum((y_pred_scores >= th) & (y_true_binary == 1))
        fp = np.sum((y_pred_scores >= th) & (y_true_binary == 0))
        precision = tp / (tp + fp) if (tp + fp) > 0 else 1.0
        recall = tp / total_positives
        precisions.append(precision)
        recalls.append(recall)
    precisions.append(0.0)
    recalls.append(1.0)
    sorted_pairs = sorted(zip(recalls, precisions))
    r_sorted, p_sorted = zip(*sorted_pairs)
    try:
        return float(np.trapezoid(p_sorted, r_sorted))
    except AttributeError:
        return float(np.trapz(p_sorted, r_sorted))

def find_data_file(filename):
    possible_paths = [
        filename,
        os.path.join('.', filename),
        os.path.join('..', filename),
        os.path.join('Weikun', filename),
        os.path.join('..', 'Weikun', filename),
        os.path.join('Ayaan', filename),
        os.path.join('..', 'Ayaan', filename),
        os.path.join('nj_daily_feature_outputs', filename),
        os.path.join('Weikun', 'nj_daily_feature_outputs', filename),
        os.path.join('njny_commute_outputs', filename),
        os.path.join('Weikun', 'njny_commute_outputs', filename)
    ]
    for p in possible_paths:
        if os.path.exists(p):
            return p
    for root, dirs, files in os.walk('.'):
        if filename in files:
            return os.path.join(root, filename)
    parent_dir = os.path.abspath(os.path.join('.', '..'))
    for root, dirs, files in os.walk(parent_dir):
        if filename in files:
            return os.path.join(root, filename)
    return None

def load_or_generate_dataset():
    feat_path = find_data_file('nj_daily_features.csv.gz') or find_data_file('final_ml_feature_matrix.csv')
    commute_path = find_data_file('nj_commuter_matrix.csv')
    
    if feat_path and os.path.exists(feat_path):
        print(f"📖 Found feature matrix at: '{feat_path}'")
        df_feat = pd.read_csv(feat_path, compression='gzip' if feat_path.endswith('.gz') else None)
        
        # Auto-map ZIP column
        zip_cols = ['zip_code', 'zcta', 'ZCTA', 'zip', 'ZIP', 'zipcode', 'GEOID', 'geoid', 'fips']
        found_zip = next((c for c in zip_cols if c in df_feat.columns), None)
        if not found_zip:
            found_zip = next((c for c in df_feat.columns if 'zip' in c.lower() or 'zcta' in c.lower()), None)
        if found_zip and found_zip != 'zip_code':
            df_feat = df_feat.rename(columns={found_zip: 'zip_code'})
            
        # Auto-map Date column
        date_cols = ['date', 'Date', 'datetime', 'time', 'TIME']
        found_date = next((c for c in date_cols if c in df_feat.columns), None)
        if not found_date:
            found_date = next((c for c in df_feat.columns if 'date' in c.lower() or 'time' in c.lower()), None)
        if found_date and found_date != 'date':
            df_feat = df_feat.rename(columns={found_date: 'date'})
            
        # Auto-map Cases column
        case_cols = ['cases', 'new_cases', 'daily_cases', 'confirmed_cases', 'case_count', 'covid_cases', 'cases_lag0']
        found_case = next((c for c in case_cols if c in df_feat.columns), None)
        if not found_case:
            found_case = next((c for c in df_feat.columns if 'case' in c.lower() or 'count' in c.lower()), None)
        if found_case and found_case != 'cases':
            df_feat = df_feat.rename(columns={found_case: 'cases'})

        df_feat['date'] = pd.to_datetime(df_feat['date'])
        df_feat['zip_code'] = df_feat['zip_code'].astype(str).str.zfill(5)
        
        # Fallback generators for missing features
        if 'cases_lag1' not in df_feat.columns:
            df_feat['cases_lag1'] = df_feat.groupby('zip_code')['cases'].shift(1)
        if 'cases_lag7' not in df_feat.columns:
            df_feat['cases_lag7'] = df_feat.groupby('zip_code')['cases'].shift(7)
        if 'case_velocity' not in df_feat.columns:
            df_feat['case_velocity'] = df_feat.groupby('zip_code')['cases'].diff().fillna(0)
        if 'case_acceleration' not in df_feat.columns:
            df_feat['case_acceleration'] = df_feat.groupby('zip_code')['case_velocity'].diff().fillna(0)
        if 'seir_R_t' not in df_feat.columns:
            roll7 = df_feat.groupby('zip_code')['cases'].transform(lambda x: x.rolling(7, min_periods=1).mean())
            roll7_prev = df_feat.groupby('zip_code')['cases'].transform(lambda x: x.shift(7).rolling(7, min_periods=1).mean())
            df_feat['seir_R_t'] = np.where(roll7_prev > 0, roll7 / roll7_prev, 1.0)
            df_feat['seir_R_t'] = np.clip(df_feat['seir_R_t'], 0.2, 4.0)
        if 'spatial_lag_cases_lag1' not in df_feat.columns:
            df_feat['spatial_lag_cases_lag1'] = df_feat['cases_lag1'] * 0.85
        if 'commuting_weighted_covid_pressure' not in df_feat.columns:
            df_feat['commuting_weighted_covid_pressure'] = df_feat['spatial_lag_cases_lag1'] * 1.25
        if 'est_ed_visits' not in df_feat.columns:
            df_feat['est_ed_visits'] = df_feat['cases'] * 0.12
        if 'pop_density_sqmi' not in df_feat.columns:
            df_feat['pop_density_sqmi'] = 12000.0
        if 'svi_score' not in df_feat.columns:
            df_feat['svi_score'] = 0.5
            
        df_feat = df_feat.dropna().reset_index(drop=True)

        unique_zips = sorted(df_feat['zip_code'].unique())
        N = len(unique_zips)
        zip_map = {z: i for i, z in enumerate(unique_zips)}
        if commute_path and os.path.exists(commute_path):
            print(f"🗺️ Ingesting LODES commuter matrix from: '{commute_path}'")
            w_df = pd.read_csv(commute_path, index_col=0)
            W = w_df.values
            row_sums = W.sum(axis=1, keepdims=True)
            W = np.divide(W, row_sums, out=np.zeros_like(W), where=row_sums!=0)
        else:
            np.random.seed(101)
            raw_W = np.random.uniform(0.01, 0.15, size=(N, N))
            np.fill_diagonal(raw_W, 0.0)
            W = raw_W / raw_W.sum(axis=1, keepdims=True)
        return df_feat, W, zip_map
    else:
        print("ℹ️ Cleaned feature file not found locally. Generating synthetic dataset...")
        num_zips, num_days = 20, 365
        nj_zips = [f'07{i:03d}' for i in range(100, 100 + num_zips)]
        dates = pd.date_range('2021-01-01', periods=num_days, freq='D')
        records = []
        for z_idx, z in enumerate(nj_zips):
            base_pop = np.random.randint(15000, 75000)
            density = np.random.uniform(5000, 35000)
            svi = np.random.uniform(0.15, 0.85)
            t = np.arange(num_days)
            wave1 = 45 * np.exp(-((t - 60)**2) / (2 * 15**2))
            wave2 = 80 * np.exp(-((t - 220)**2) / (2 * 25**2))
            baseline = 5 + np.random.poisson(3, num_days)
            cases = np.maximum(0, (wave1 + wave2 + baseline) * (1 + 0.1 * z_idx)).astype(int)
            for d, c in zip(dates, cases):
                records.append({
                    'zip_code': z, 'date': d, 'cases': c, 'population': base_pop,
                    'pop_density_sqmi': density, 'svi_score': svi
                })
        df = pd.DataFrame(records).sort_values(['zip_code', 'date'])
        N = num_zips
        zip_map = {z: i for i, z in enumerate(nj_zips)}
        np.random.seed(101)
        raw_W = np.random.uniform(0.01, 0.15, size=(N, N))
        np.fill_diagonal(raw_W, 0.0)
        W = raw_W / raw_W.sum(axis=1, keepdims=True)
        df['cases_lag1'] = df.groupby('zip_code')['cases'].shift(1)
        df['cases_lag7'] = df.groupby('zip_code')['cases'].shift(7)
        df['case_velocity'] = df.groupby('zip_code')['cases'].diff().fillna(0)
        df['case_acceleration'] = df.groupby('zip_code')['case_velocity'].diff().fillna(0)
        roll7 = df.groupby('zip_code')['cases'].transform(lambda x: x.rolling(7, min_periods=1).mean())
        roll7_prev = df.groupby('zip_code')['cases'].transform(lambda x: x.shift(7).rolling(7, min_periods=1).mean())
        df['seir_R_t'] = np.where(roll7_prev > 0, roll7 / roll7_prev, 1.0)
        df['seir_R_t'] = np.clip(df['seir_R_t'], 0.2, 4.0)
        df['est_ed_visits'] = df['cases'] * 0.12
        pivot_lag1 = df.pivot(index='date', columns='zip_code', values='cases_lag1').fillna(0)
        spatial_lag_matrix = pivot_lag1.values @ W.T
        spatial_lag_df = pd.DataFrame(spatial_lag_matrix, index=pivot_lag1.index, columns=pivot_lag1.columns)
        spatial_lag_long = spatial_lag_df.reset_index().melt(id_vars='date', var_name='zip_code', value_name='spatial_lag_cases_lag1')
        df = df.merge(spatial_lag_long, on=['date', 'zip_code'], how='left')
        df['commuting_weighted_covid_pressure'] = df['spatial_lag_cases_lag1'] * 1.25
        df_clean = df.dropna().reset_index(drop=True)
        return df_clean, W, zip_map



In [9]:
feature_df, W_matrix, zip_to_idx = load_or_generate_dataset()
feature_cols = ['cases_lag1', 'cases_lag7', 'spatial_lag_cases_lag1', 'commuting_weighted_covid_pressure', 'seir_R_t', 'case_velocity', 'case_acceleration', 'est_ed_visits', 'pop_density_sqmi', 'svi_score']
print(f'Loaded dataset: {len(feature_df)} rows across {len(zip_to_idx)} ZIP codes.')


📖 Found feature matrix at: 'nj_daily_features.csv.gz'
Loaded dataset: 808511 rows across 584 ZIP codes.


In [10]:
# Fit Random Forest Baseline
X = feature_df[feature_cols].values
y = feature_df['cases'].values

rf_model = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf_model.fit(X, y)

preds = rf_model.predict(X)
rmse = calc_rmse(y, preds)
mae = calc_mae(y, preds)

print('=== Random Forest Baseline Model Fit ===')
print(f'RMSE: {rmse:.4f} cases')
print(f'MAE:  {mae:.4f} cases')

# Feature Importances
importances = rf_model.feature_importances_
importance_df = pd.DataFrame({
    'Feature': feature_cols,
    'RF Importance': importances
}).sort_values('RF Importance', ascending=False)

print('=== Random Forest Feature Importances ===')
print(importance_df.to_string(index=False))

# Plot Feature Importances
plt.figure(figsize=(10, 5))
plt.barh(importance_df['Feature'][::-1], importance_df['RF Importance'][::-1], color='#2b5c8f')
plt.title('Random Forest Feature Importance Rankings')
plt.xlabel('Importance Score')
plt.tight_layout()
os.makedirs('rf_baseline_outputs', exist_ok=True)
plt.savefig('rf_baseline_outputs/rf_feature_importances.png', dpi=150)
plt.close()

importance_df.to_csv('rf_baseline_outputs/rf_feature_importances.csv', index=False)
print('✅ Exported feature importances to rf_baseline_outputs/rf_feature_importances.csv')


=== Random Forest Baseline Model Fit ===
RMSE: 0.0059 cases
MAE:  0.0000 cases
=== Random Forest Feature Importances ===
                          Feature  RF Importance
                    est_ed_visits       0.999966
                         seir_R_t       0.000016
                    case_velocity       0.000006
                case_acceleration       0.000004
                       cases_lag7       0.000003
           spatial_lag_cases_lag1       0.000002
commuting_weighted_covid_pressure       0.000002
                       cases_lag1       0.000001
                 pop_density_sqmi       0.000000
                        svi_score       0.000000
✅ Exported feature importances to rf_baseline_outputs/rf_feature_importances.csv
